In [1]:
# ============================================================
# DATASET: LendingClub
# Purpose: targeted hyperparameter investigation of RF's collapse.
#          Testing class_weight='balanced' as an alternative to
#          resampling entirely - a different mechanism for handling
#          imbalance (reweights the loss function rather than
#          changing the training data itself)
# ============================================================

import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from imblearn.metrics import geometric_mean_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
import re

SCALED_DIR = Path("../Data/Processed")
RANDOM_SEED = 42

def clean_column_names(df):
    df.columns = [re.sub(r"[\[\]<>]", "_", str(col)) for col in df.columns]
    return df

X_lending_train = pd.read_csv(SCALED_DIR / "lending_X_train.csv")
clean_column_names(X_lending_train)
X_lending_test = pd.read_csv(SCALED_DIR / "lending_X_test.csv")
clean_column_names(X_lending_test)
y_lending_train = pd.read_csv(SCALED_DIR / "lending_y_train.csv").squeeze()
y_lending_test = pd.read_csv(SCALED_DIR / "lending_y_test.csv").squeeze()

def evaluate_model(model, X_train, y_train, X_test, y_test, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "config": name,
        "AUC-ROC": roc_auc_score(y_test, y_proba),
        "F1": f1_score(y_test, y_pred),
        "G-mean": geometric_mean_score(y_test, y_pred),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "positive_predictions": int(y_pred.sum())
    }

# RF with class_weight='balanced', no resampling at all
rf_balanced = RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200, class_weight="balanced")
result_rf_balanced = evaluate_model(rf_balanced, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "RF (class_weight=balanced, no resampling)")

print(pd.DataFrame([result_rf_balanced]))

                                      config   AUC-ROC        F1    G-mean  \
0  RF (class_weight=balanced, no resampling)  0.826672  0.105263  0.235702   

        MCC  positive_predictions  
0  0.233688                     2  


In [2]:
# ============================================================
# DATASET: LendingClub
# Purpose: test whether constraining tree complexity (rather than
#          adjusting for class imbalance directly) changes RF's
#          behaviour - addresses potential overfitting rather than
#          the class-weighting/resampling angle already tested
# ============================================================

rf_constrained = RandomForestClassifier(
    random_state=RANDOM_SEED, n_estimators=200,
    max_depth=5, min_samples_leaf=10  # meaningfully shallower/more conservative than default
)
result_rf_constrained = evaluate_model(rf_constrained, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "RF (constrained depth, no resampling)")

# Same constraints, combined with SMOTE-ENN (the best-performing resampling
# technique found so far), to see if constraining complexity helps THIS
# specific combination succeed where the default-depth version failed
rf_constrained_smoteenn = ImbPipeline([
    ("smoteenn", SMOTEENN(random_state=RANDOM_SEED)),
    ("clf", RandomForestClassifier(random_state=RANDOM_SEED, n_estimators=200, max_depth=5, min_samples_leaf=10))
])
result_rf_constrained_smoteenn = evaluate_model(rf_constrained_smoteenn, X_lending_train, y_lending_train, X_lending_test, y_lending_test, "RF (constrained depth, SMOTE-ENN)")

print(pd.DataFrame([result_rf_constrained, result_rf_constrained_smoteenn]))

                                  config   AUC-ROC        F1   G-mean  \
0  RF (constrained depth, no resampling)  0.793477  0.000000  0.00000   
1      RF (constrained depth, SMOTE-ENN)  0.701148  0.077273  0.61577   

        MCC  positive_predictions  
0  0.000000                     0  
1  0.091122                   404  
